In [1]:
logs = """
2025-10-10,12:01:32,192.168.1.2,GET,/index.html,200,1024
2025-10-10,12:01:33,192.168.1.3,GET,/products.html,200,850
2025-10-10,12:01:35,192.168.1.4,GET,/contact.html,404,512
2025-10-10,12:01:38,192.168.1.5,POST,/checkout,500,128
2025-10-10,12:01:41,192.168.1.6,GET,/index.html,200,1024
2025-10-10,12:01:45,192.168.1.7,GET,/images/logo.png,200,256
2025-10-10,12:01:48,192.168.1.8,GET,/about.html,404,512
2025-10-10,12:01:53,192.168.1.9,POST,/login,403,64
2025-10-10,12:02:01,192.168.1.10,GET,/index.html,200,1024
2025-10-10,12:02:07,192.168.1.11,POST,/checkout,500,128
2025-10-10,12:02:12,192.168.1.12,GET,/contact.html,404,512
2025-10-10,12:02:15,192.168.1.13,GET,/index.html,200,1024
2025-10-10,12:02:21,192.168.1.14,GET,/products.html,200,850
2025-10-10,12:02:23,192.168.1.15,GET,/about.html,404,512
2025-10-10,12:02:29,192.168.1.16,POST,/checkout,500,128
2025-10-10,12:02:31,192.168.1.17,GET,/images/logo.png,200,256
2025-10-10,12:02:34,192.168.1.18,GET,/contact.html,404,512
2025-10-10,12:02:38,192.168.1.19,POST,/login,403,64
2025-10-10,12:02:41,192.168.1.20,GET,/index.html,200,1024
2025-10-10,12:02:47,192.168.1.21,GET,/products.html,200,850
"""


In [2]:
def mapper(line):
    fields = line.strip().split(",")

    # Ensure valid log format
    if len(fields) < 7:
        return []

    status = fields[5]   # the status code is the 6th field
    return [(status, 1)]


In [3]:
mapped = []

for line in logs.strip().split("\n"):
    mapped.extend(mapper(line))

mapped


[('200', 1),
 ('200', 1),
 ('404', 1),
 ('500', 1),
 ('200', 1),
 ('200', 1),
 ('404', 1),
 ('403', 1),
 ('200', 1),
 ('500', 1),
 ('404', 1),
 ('200', 1),
 ('200', 1),
 ('404', 1),
 ('500', 1),
 ('200', 1),
 ('404', 1),
 ('403', 1),
 ('200', 1),
 ('200', 1)]

In [4]:
from collections import defaultdict

grouped = defaultdict(list)

for status, value in mapped:
    grouped[status].append(value)

grouped


defaultdict(list,
            {'200': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
             '404': [1, 1, 1, 1, 1],
             '500': [1, 1, 1],
             '403': [1, 1]})

In [5]:
def reducer(status, values):
    return (status, sum(values))


In [6]:
reduced = []

for status, values in grouped.items():
    reduced.append(reducer(status, values))

# Sort numerically by status code
reduced = sorted(reduced, key=lambda x: int(x[0]))

for status, count in reduced:
    print(f"HTTP {status}: {count} requests")


HTTP 200: 10 requests
HTTP 403: 2 requests
HTTP 404: 5 requests
HTTP 500: 3 requests


In [7]:
def mapper_url(line):
    fields = line.strip().split(",")
    if len(fields) < 7:
        return []
    url = fields[4]
    return [(url, 1)]


In [8]:
def mapper_size(line):
    fields = line.strip().split(",")
    if len(fields) < 7:
        return []
    status = fields[5]
    size = int(fields[6])
    return [(status, size)]


In [9]:
def mapper_errors_only(line):
    fields = line.strip().split(",")
    if len(fields) < 7:
        return []
    status = fields[5]
    if status == "200":
        return []
    return [(status, 1)]
